In [69]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
import os
from sentence_transformers import CrossEncoder
from gptcache import cache
from gptcache.manager import (
CacheBase, VectorBase, get_data_manager)
from gptcache.config import Config
from gptcache.similarity_evaluation import (
SearchDistanceEvaluation)
from gptcache.embedding import Onnx
from gptcache.adapter.api import get, put
from gptcache.processor.pre import get_prompt


In [70]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1970.42it/s]


In [71]:
def gptcache_embedding(text, **_):
    """Return the 384-dimension vector GPTCache expects."""
    return embedder.embed_query(text)


cache.init(
    pre_embedding_func=get_prompt,
    embedding_func=gptcache_embedding,

    data_manager=get_data_manager(
        CacheBase("sqlite"),
        VectorBase(
            "faiss",
            dimension=384
        )
    ),

    similarity_evaluation=SearchDistanceEvaluation(),

    config=Config(
        similarity_threshold=0.85
    )
)

In [72]:
load_dotenv()

True

In [53]:
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.7)

In [54]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [55]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [56]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [57]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [58]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [59]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [60]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [61]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [62]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [63]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
        
        
    }
)



In [64]:
result=rag_chain | rag_prompt | llm


In [65]:
op=result.invoke("What is summary of module?")

In [66]:
print(print(op.content))

**Summary of the modules described in the provided context**

- **Module 2 – Cryptography Fundamentals**  
  Introduces the core concepts of cryptography used in blockchain, including hashing techniques, public‑ and private‑key encryption, and digital signatures.

- **Module 3 – Consensus Algorithms**  
  Explains how distributed networks achieve agreement, covering Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT) mechanisms.

- **Module 4 – Smart Contracts and Ethereum**  
  Teaches how to write, test, and deploy smart contracts on the Ethereum platform using the Solidity programming language and the Remix integrated development environment (IDE).

- **Module 5 – Blockchain and AI Integration**  
  Explores the intersection of blockchain and artificial intelligence, focusing on using blockchain to secure the sharing of AI models and to ensure provenance and integrity of training data.
None


In [73]:


def cached_rag_chain(question):

    # -------------------------
    # 1. Check semantic cache
    # -------------------------
    cached_result = get(question)

    if cached_result is not None:
        print("✅ CACHE HIT")
        return cached_result

    print("❌ CACHE MISS")

    # -------------------------
    # 2. Run RAG
    # -------------------------
    response = result.invoke(question)

    # -------------------------
    # 3. Extract answer
    # -------------------------
    answer = response.content

    # -------------------------
    # 4. Store in GPTCache
    # -------------------------
    put(question, answer)

    return answer

In [75]:
pred=cached_rag_chain("Can you provide a summary of the module?")

✅ CACHE HIT


In [76]:
print(pred)

**Summary of the modules described in the provided context**

- **Module 2 – Cryptography Fundamentals**  
  Covers the basics of cryptographic techniques that secure blockchain systems, including hashing functions, public‑ and private‑key encryption, and digital signatures.

- **Module 3 – Consensus Algorithms**  
  Explains how blockchain networks achieve agreement on the state of the ledger, focusing on Proof of Work (PoW), Proof of Stake (PoS), and Byzantine Fault Tolerance (BFT) mechanisms.

- **Module 4 – Smart Contracts and Ethereum**  
  Teaches how to write, test, and deploy smart contracts on the Ethereum platform using the Solidity programming language and the Remix IDE.

- **Module 5 – Blockchain and AI Integration**  
  Discusses the intersection of blockchain and artificial intelligence, emphasizing how blockchain can be used to protect AI model sharing and ensure the provenance of training data.


In [37]:
import gptcache

print(gptcache.__version__)
print(type(cache))
print(dir(cache))


0.1.44
<class 'gptcache.core.Cache'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'cache_enable_func', 'config', 'data_manager', 'embedding_func', 'flush', 'has_init', 'import_data', 'init', 'next_cache', 'post_process_messages_func', 'pre_embedding_func', 'report', 'set_azure_openai_key', 'set_openai_key', 'similarity_evaluation']
